# AI Agent 기반 DB → FHIR R4 단계별 테스트

현재 TASK의 핵심인 LLM SQL, Source Row Contract, 결정적 Resource Renderer와 검증 파이프라인을 순서대로 확인합니다. FHIR Resource를 row 단위로 생성하는 LLM 단계는 없습니다.

In [1]:
from collections import Counter
from dataclasses import replace
from pathlib import Path
import json
import sys
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "main.py").is_file():
    candidates = [p for p in PROJECT_ROOT.parents if (p / "main.py").is_file()]
    if not candidates:
        raise RuntimeError("main.py가 있는 프로젝트 루트에서 실행하세요.")
    PROJECT_ROOT = candidates[0]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# 외부 시스템을 사용하거나 결과를 저장하는 단계는 필요할 때만 켭니다.
RUN_DB_CHECK = True
RUN_VECTOR_REBUILD = False
RUN_RAG_SEARCH = True
RUN_LLM_SQL = False
RUN_SINGLE_PATIENT_PIPELINE = False
RUN_EXTERNAL_VALIDATOR = True
RUN_BATCH_10 = True

NOTEBOOK_OUTPUT = PROJECT_ROOT / "fhir_export_notebook_test"
print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: /home/jhkim/fhir_agent


## 1. 설정과 Resource Definition

SQL Agent, Vector DB, 결정적 Renderer와 Validator 설정을 로드합니다.

In [2]:
from fhir_agent_pipeline.config import load_settings
from fhir_agent_pipeline.resource_definition import ordered_definitions

settings = load_settings(require_database=False)
assert settings.source_profile_path.is_file()
assert settings.fhir_schema_path.is_file()

selected_definitions = ordered_definitions(settings.fhir_output_resources)
definitions = {
    definition.resource_type: definition.as_dict()
    for definition in selected_definitions
}
assert set(definitions) == set(settings.fhir_output_resources)
assert all(x["generation_mode"] == "deterministic_renderer" for x in definitions.values())
display({
    "source_profile": str(settings.source_profile_path),
    "fhir_schema_path": str(settings.fhir_schema_path),
    "output_resources": settings.fhir_output_resources,
    "resource_definitions": definitions,
})

{'source_profile': '/home/jhkim/fhir_agent/profiles/mimic_iv/profile.json',
 'fhir_schema_path': '/home/jhkim/fhir_agent/db/fhir.schema.json',
 'output_resources': ['Patient',
  'Encounter',
  'Condition',
  'Procedure',
  'Observation'],
 'resource_definitions': {'Patient': {'resource_type': 'Patient',
   'order': 0,
   'required_source_aliases': ('_patient_key',),
   'optional_source_aliases': ('_gender', '_death_datetime'),
   'fhir_mapping_policy': {'id': 'stable id from _patient_key',
    'gender': 'normalized from _gender when present',
    'deceasedDateTime': 'from _death_datetime when present'},
   'dependencies': (),
   'required_fields': ('id',),
   'source_required': True,
   'bundle_required': True,
   'identity_alias': '_patient_key',
   'encounter_scoped': False,
   'query_limit_setting': None,
   'query_limit_per_encounter': False,
   'page_size': 500,
   'pagination_enabled': False,
   'fixed_status': None,
   'closed_status': None,
   'open_status': None,
   'generatio

## 2. 결정적 Observation Renderer

외부 DB나 LLM 없이 source alias 한 행이 Observation 한 건으로 변환되고 LLM 호출 수가 0인지 확인합니다.

In [3]:
from fhir_agent_pipeline.fhir_schema import get_fhir_schema
from fhir_agent_pipeline.resource_renderer import DeterministicResourceRenderer

local_settings = replace(settings, vectors_enabled=False, fhir_validator_jar=None)
renderer = DeterministicResourceRenderer(
    local_settings, get_fhir_schema(settings.fhir_schema_path)
)
sample_rows = [{
    "_patient_key": "P-1",
    "_encounter_key": "E-1",
    "_resource_key": "O-1",
    "_code": "LAB-1",
    "_event_datetime": "2026-07-28 12:00:00",
    "_code_system": None,
    "_display": "Example lab",
    "_value_number": "2.5",
    "_value_text": None,
    "_unit": "mg",
    "_unit_system": None,
    "_unit_code": None,
    "_reference_low": "1.0",
    "_reference_high": "3.0",
    "_interpretation": "normal",
}]
rendered = renderer.generate("Observation", sample_rows, "P-1")
assert len(rendered.resources) == len(sample_rows)
assert rendered.metrics()["llm_calls_used"] == 0
assert rendered.resources[0]["valueQuantity"]["value"] == 2.5
display({"metrics": rendered.metrics(), "resource": rendered.resources[0]})

{'metrics': {'source_rows': 1,
  'resources_rendered': 1,
  'count_alignment_rate': 1.0,
  'llm_calls_used': 0},
 'resource': {'resourceType': 'Observation',
  'id': 'O-1',
  'subject': {'reference': 'Patient/P-1'},
  'code': {'coding': [{'code': 'LAB-1'}], 'text': 'Example lab'},
  'encounter': {'reference': 'Encounter/E-1'},
  'status': 'final',
  'effectiveDateTime': '2026-07-28T12:00:00Z',
  'valueQuantity': {'value': 2.5, 'unit': 'mg'},
  'referenceRange': [{'low': {'value': 1}, 'high': {'value': 3}}],
  'interpretation': [{'text': 'normal'}]}}

## 3. Cohort와 Source DB metadata

Cohort provider가 구성된 경우 patient key를 읽습니다. DB catalog 검사는 `RUN_DB_CHECK=True`일 때만 실행합니다.

In [4]:
from fhir_agent_pipeline.export_service import iter_cohort_patient_ids

patient_keys = []
if settings.source_data_path.exists():
    try:
        patient_keys = list(iter_cohort_patient_ids(settings, limit=10))
    except (FileNotFoundError, ValueError) as exc:
        print("cohort 건너뜀:", exc)
PATIENT_KEY = patient_keys[0] if patient_keys else settings.source_patient_id
display({"patient_keys": patient_keys, "selected_patient_key": PATIENT_KEY})

if RUN_DB_CHECK:
    from fhir_agent_pipeline.db import build_engine, SourceSchemaCatalog
    db_settings = load_settings()
    engine = build_engine(db_settings)
    try:
        catalog = SourceSchemaCatalog.load(db_settings, engine)
        assert catalog.tables
        assert all(t.schema.lower() not in {"public", "information_schema"} for t in catalog.tables)
        display({
            "schemas": sorted(catalog.schemas),
            "table_count": len(catalog.tables),
            "sample_tables": [t.qualified_name for t in catalog.tables[:10]],
        })
    finally:
        engine.dispose()

{'patient_keys': [10007795,
  10007928,
  10009628,
  10011398,
  10004457,
  10003046,
  10014078,
  10003400,
  10020187,
  10038999],
 'selected_patient_key': 10007795}

{'schemas': ['mimiciv_derived', 'mimiciv_hosp', 'mimiciv_icu'],
 'table_count': 59,
 'sample_tables': ['mimiciv_derived.antibiotic',
  'mimiciv_derived.bg',
  'mimiciv_derived.chemistry',
  'mimiciv_derived.complete_blood_count',
  'mimiciv_derived.crrt',
  'mimiciv_derived.dobutamine',
  'mimiciv_derived.dopamine',
  'mimiciv_derived.enzyme',
  'mimiciv_derived.epinephrine',
  'mimiciv_derived.gcs']}

## 4. FHIR/Source Schema Vector DB

Vector rebuild와 RAG 검색은 명시적으로 플래그를 켰을 때만 수행합니다.

In [5]:
if RUN_VECTOR_REBUILD or RUN_RAG_SEARCH:
    db_settings = load_settings()
    from fhir_agent_pipeline.vector_store import PgVectorKnowledgeStore, build_knowledge_bases

    if RUN_VECTOR_REBUILD:
        display(build_knowledge_bases(db_settings))

    store = PgVectorKnowledgeStore(db_settings)
    try:
        reports = [
            store.inspect_table(db_settings.sql_vector_table),
            store.inspect_table(db_settings.fhir_vector_table),
        ]
        assert all(x["exists"] and x["compatible"] and x["rows"] > 0 for x in reports)
        display(reports)
        if RUN_RAG_SEARCH:
            sql_matches = store.search(
                db_settings.sql_vector_table,
                "patient encounter observation columns primary key foreign key",
                k=5,
                profile_scoped=True,
            )
            fhir_matches = store.search(
                db_settings.fhir_vector_table,
                "FHIR R4 Observation code effective value referenceRange",
                k=5,
                profile_scoped=False,
            )
            assert sql_matches and fhir_matches
            display({"sql_matches": sql_matches, "fhir_matches": fhir_matches})
    finally:
        store.engine.dispose()
else:
    print("Vector 단계 건너뜀")

[{'table': 'public.source_sql_schema_vectors',
  'exists': True,
  'columns': {'id': 'bigint',
   'content': 'text',
   'metadata': 'jsonb',
   'embedding': 'vector(768)',
   'content_hash': 'character varying(64)',
   'updated_at': 'timestamp with time zone'},
  'rows': 59,
  'compatible': True,
  'indexes': ['source_sql_schema_vectors_content_hash_key',
   'source_sql_schema_vectors_embedding_hnsw_39602121',
   'source_sql_schema_vectors_pkey'],
  'current_profile_rows': 59,
  'global_rows': 0},
 {'table': 'public.fhir_schema_vectors',
  'exists': True,
  'columns': {'id': 'bigint',
   'content': 'text',
   'metadata': 'jsonb',
   'embedding': 'vector(768)',
   'content_hash': 'character varying(64)',
   'updated_at': 'timestamp with time zone'},
  'rows': 11499,
  'compatible': True,
  'indexes': ['fhir_schema_vectors_content_hash_key',
   'fhir_schema_vectors_embedding_hnsw_39590601',
   'fhir_schema_vectors_pkey'],
  'current_profile_rows': 0,
  'global_rows': 11499}]

{'sql_matches': [{'content': 'PostgreSQL table mimiciv_hosp.patients\nColumns: subject_id INTEGER PRIMARY KEY, gender CHAR(1), anchor_age SMALLINT, anchor_year SMALLINT, anchor_year_group VARCHAR(20), dod DATE\nForeign keys: none',
   'metadata': {'kind': 'table_schema',
    'table': 'patients',
    'schema': 'mimiciv_hosp',
    'source': 'postgresql',
    'profile': 'mimic-iv-2.2',
    'profile_fingerprint': '34765ebe64e843e245e72e5664d19411de347c0c9250e3888138e2c1e7b2188c'},
   'similarity': 0.7132009018177929},
  {'content': 'PostgreSQL table mimiciv_hosp.d_icd_diagnoses\nColumns: icd_code CHAR(7) PRIMARY KEY, icd_version SMALLINT PRIMARY KEY, long_title VARCHAR(255)\nForeign keys: none',
   'metadata': {'kind': 'table_schema',
    'table': 'd_icd_diagnoses',
    'schema': 'mimiciv_hosp',
    'source': 'postgresql',
    'profile': 'mimic-iv-2.2',
    'profile_fingerprint': '34765ebe64e843e245e72e5664d19411de347c0c9250e3888138e2c1e7b2188c'},
   'similarity': 0.7129506728200213},
  {'

## 5. LLM SQL → Safe SQL Runtime

선택 환자의 Patient SQL을 생성하고 alias, read-only 실행 결과와 retry trace를 확인합니다.

In [6]:
queried = None
if RUN_LLM_SQL:
    if PATIENT_KEY is None:
        raise ValueError("실행할 PATIENT_KEY가 없습니다.")
    from fhir_agent_pipeline.llm_query_service import LlmClinicalQueryService

    query_service = LlmClinicalQueryService(load_settings())
    try:
        queried = query_service.query("Patient", PATIENT_KEY, [], None)
        assert len(queried.rows) == 1
        assert {"_patient_key", "_gender", "_death_datetime"} <= set(queried.rows[0])
        display({
            "sql": queried.trace.sql,
            "row_count": len(queried.rows),
            "llm_calls": queried.trace.llm_calls_used,
            "failure_categories": queried.trace.failure_categories,
            "rag_used": queried.trace.rag_used,
        })
    finally:
        query_service.close()
else:
    print("LLM SQL 단계 건너뜀")

LLM SQL 단계 건너뜀


## 6. 단일 환자 전체 파이프라인

모든 LLM 호출은 SQL 단계에서만 발생하며 `rendering_metrics`의 LLM 호출 수는 항상 0이어야 합니다.

In [7]:
pipeline_result = None
if RUN_SINGLE_PATIENT_PIPELINE:
    if PATIENT_KEY is None:
        raise ValueError("실행할 PATIENT_KEY가 없습니다.")
    from fhir_agent_pipeline.llm_pipeline import LlmSqlToFhirPipeline

    pipeline_result = LlmSqlToFhirPipeline(load_settings()).run(PATIENT_KEY)
    resource_counts = Counter(x["resourceType"] for x in pipeline_result.generated_resources)
    observations = [
        resource
        for resource in pipeline_result.generated_resources
        if resource["resourceType"] == "Observation"
    ]
    observation_scope_counts = {
        "with_encounter": sum("encounter" in x for x in observations),
        "without_encounter": sum("encounter" not in x for x in observations),
    }
    clinical_counts_by_encounter = {}
    for resource in pipeline_result.generated_resources:
        reference = resource.get("encounter", {}).get("reference")
        if not reference:
            continue
        encounter_key = reference.split("/", 1)[-1]
        counts = clinical_counts_by_encounter.setdefault(encounter_key, Counter())
        counts[resource["resourceType"]] += 1
    clinical_counts_by_encounter = {
        key: dict(counts)
        for key, counts in sorted(clinical_counts_by_encounter.items())
    }
    assert pipeline_result.validation_report.valid
    assert pipeline_result.task_evaluation["task_quality_pass"]
    assert all(
        metric["llm_calls_used"] == 0
        for metric in pipeline_result.rendering_metrics.values()
    )
    display({
        "mode": pipeline_result.trace_dict()["mode"],
        "resource_counts": dict(resource_counts),
        "source_row_counts": pipeline_result.source_row_counts,
        "observation_scope_counts": observation_scope_counts,
        "clinical_counts_by_encounter": clinical_counts_by_encounter,
        "sql_llm_calls": pipeline_result.llm_calls_used,
        "rendering_metrics": pipeline_result.rendering_metrics,
    })
else:
    print("단일 환자 파이프라인 건너뜀")

단일 환자 파이프라인 건너뜀


## 7. Bundle page와 Validation

Renderer sample로 page builder를 확인하고, 전체 파이프라인을 실행한 경우 생성 Bundle을 검증합니다.

In [8]:
from fhir_agent_pipeline.fhir_models import build_bundle_pages

sample_observations = [
    {**rendered.resources[0], "id": f"O-{index}"}
    for index in range(5)
]
sample_pages = build_bundle_pages(sample_observations, observation_page_size=2)
page_entry_counts = [len(page.get("entry", [])) for page in sample_pages]
assert page_entry_counts == [0, 2, 2, 1]
display({"sample_page_entry_counts": page_entry_counts})

if pipeline_result is not None:
    validation_report = pipeline_result.validation_report
    assert validation_report is not None
    assert validation_report.valid, validation_report.errors

    if RUN_EXTERNAL_VALIDATOR:
        assert validation_report.external_validator_used
        issue_counts = validation_report.validator_issue_counts
        assert issue_counts.get("fatal", 0) == 0
        assert issue_counts.get("error", 0) == 0

    display(validation_report.as_dict())

{'sample_page_entry_counts': [0, 2, 2, 1]}

## 8. 선택적 cohort 10명 batch

결과 파일을 생성하므로 `RUN_BATCH_10=True`일 때만 수행합니다.

In [9]:
if RUN_BATCH_10:
    from fhir_agent_pipeline.export_service import ExportRequest, FhirBatchExportService

    batch_settings = load_settings()
    summary = FhirBatchExportService(batch_settings).run(
        ExportRequest(output_directory=NOTEBOOK_OUTPUT, limit=10, resume=True)
    )
    assert summary["invalid_patients"] == 0
    assert summary["failed_patients"] == 0
    display(summary)
else:
    print("batch 단계 건너뜀")

{'output_directory': '/home/jhkim/fhir_agent/fhir_export_notebook_test',
 'attempted_patients': 10,
 'generated_patients': 10,
 'valid_patients': 10,
 'invalid_patients': 0,
 'failed_patients': 0,
 'resumed_patients': 0,
 'resource_counts': {'Patient': 10,
  'Encounter': 27,
  'Condition': 381,
  'Procedure': 82,
  'Observation': 8035},
 'validator_issue_counts': {'fatal': 0,
  'error': 0,
  'warning': 36018,
  'information': 0},
 'mean_task_evaluation': {'source_field_accuracy': 1.0,
  'clinical_resource_macro_f1': 1.0,
  'reference_accuracy': 1.0,
  'coding_accuracy': 1.0},
 'performance_evaluation': {'overall_task_pass_rate': 1.0,
  'strict_end_to_end_pass_rate': 1.0,
  'fhir_standard_conformance': {'evaluated_bundles': 10,
   'passed_bundles': 10,
   'fhir_validation_pass_rate': 1.0},
  'source_fidelity': {'source_field_accuracy': {'correct': 71528,
    'evaluated': 71528,
    'accuracy': 1.0}},
  'resource_generation_accuracy': {'resource_count_alignment': {'overall': {'expected':

## 판정 기준

- 모든 Resource Definition의 generation mode는 `deterministic_renderer`입니다.
- Renderer는 source row 수와 동일한 리소스를 만들고 LLM 호출 수는 0입니다.
- SQL 실행 시 required alias, patient scope와 Safe SQL 정책을 통과해야 합니다.
- 전체 파이프라인 trace mode는 `llm-sql-and-deterministic-fhir`입니다.
- Bundle은 local FHIR Schema, 구조와 reference 검증을 통과해야 합니다.
- 선택적 batch는 invalid 0, failed 0이어야 합니다.